Setting up a training loop of unsupervised HKEM without patches

This demo is a jupyter notebook, i.e. intended to be run step by step.

Author: Daniel Deidda

First version: 13th of Nov 2023

CCP SyneRBI Synergistic Image Reconstruction Framework (SIRF).
Copyright 2023 NPL.
SPDX-License-Identifier: Apache-2.0

# Setting up the training


In [1]:
# Get the parent directory
import sys
import os
import pathlib

#parent_dir = os.path.dirname(os.path.realpath('.'))
notebooks_dir = pathlib.Path().parent.absolute() 
parent_dir =  os.path.dirname(notebooks_dir)
print(parent_dir, notebooks_dir)
# Add the parent directory to sys.path

sys.path.append(parent_dir)
NEMApath='/home/dd3/AI_uncert/NEMA-DL'
sys.path.append(NEMApath)
print(NEMApath)

/home/dd3/devel/github/ARTCertainty/artcertainty /home/dd3/devel/github/ARTCertainty/artcertainty/notebooks
/home/dd3/AI_uncert/NEMA-DL


In [ ]:
# Import the PET reconstruction engine
import sirf.STIR as pet
# Set the verbosity
pet.set_verbosity(0)
# Store temporary sinograms in RAM
pet.AcquisitionData.set_storage_scheme("memory")
# SIRF STIR message redirector
import sirf

import sirf.STIR as pet
msg = sirf.STIR.MessageRedirector(info=None, warn=None, errr=None)
# Load dataset and model
from kernel.LHK import  kernelise_image, set_KOSMAPOSL
from architectures.UNet3D import  UNet as unet3d
from architectures.UNet3D import  AttentionUNet as aunet3d,  make_image_network_compatible, get_new_shape
from utils.torch_operations import   tdivide,npdivide, update_subset_model
from utils.plots import  plot_many_tensors,plot_many_numpys
from utils.sirf_torch import primal_op as F
from utils.sirf_torch import dual_op as B
from utils.system import create_working_dir_and_move_into
from utils.analytics import estimate_MSE_and_save,  plot_losses
from utils.sirf_modelling import  get_acquisition_model, get_acquisition_model_with_normacf,get_acquisition_model_real_with_norm_and_umap
from algorithms.Algorithm import Algorithm
from algorithms.neuralKEM import neuralKEM
from algorithms.Algorithm import read_simulation, get_working_dir_from_outpath
#from algorithms.Brain_simulation import make_image_simulation_2d

import os
import numpy as np
import time
import torch
from tqdm.auto import  tqdm
device = torch.device("cuda" if torch.cuda.is_available() else "cpu:1")
from numpy import inf
working_dir=create_working_dir_and_move_into(NEMApath)
print(working_dir)
sinogram_template = pet.AcquisitionData(NEMApath+'/projdata_bed1.hs');
add_sino = pet.AcquisitionData(NEMApath+'/additive_sino.hs');
norm_sino = pet.AcquisitionData(NEMApath+'/normprojdata_bed1.hs');

# normacf_sino0 = pet.AcquisitionData(NEMApath+'/normacfprojdata_bed1.hs');
# normacf_sino0_tt = torch.from_numpy(normacf_sino0.as_array()).repeat(1,1,1,1,1).to(device)

y = torch.from_numpy(sinogram_template.as_array()).repeat(1,1,1,1,1).to(device)
add_sino_tt = torch.from_numpy(add_sino.as_array()).repeat(1,1,1,1,1).to(device)
norm_sino_np = norm_sino.as_array()

# true_brain = pet.ImageData('FDG_tumour_small.hv') #(144x144)
umap = pet.ImageData(NEMApath+'/umap.hv')

umap_np = umap.as_array()
umap_np=np.nan_to_num(umap_np, nan=0.0, posinf=0.0, neginf=0.0) 
umap.fill(umap_np)
image_template = umap.clone()

# umap_tt = torch.from_numpy(umap_np).repeat(1,1,1,1,1).to(device)

# anat_tt = torch.from_numpy(umap_np/umap_np.max()).repeat(1,1,1,1,1).to(device)
anat = pet.ImageData(NEMApath+'/umap.hv')
anat_np = anat.as_array()
anat_np=np.nan_to_num(anat_np, nan=0.0, posinf=0.0, neginf=0.0) 
anat.fill(anat_np/anat_np.max())
# anat=image_template.clone().fill(umap_np/umap_np.max())
import gc

In [22]:
import os
import numpy as np
import time
import torch
from tqdm.auto import  tqdm
from architectures.UNet3D import  UNet, make_image_network_compatible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu:1")
from numpy import inf

from utils.system import create_working_dir_and_move_into
# simpath='/home/dd3/AI_uncert/NEM'
working_dir=create_working_dir_and_move_into(NEMApath)
net = UNet(1,16,1).to(device)
print(working_dir)

checkpoint = torch.load(working_dir+'/nKEM_2d0_k27_w9_ks1_0_LR0_001_seed5_net_scaled_dit150_eit4_tit'+'_trained_extra.torch_model', map_location=device)
net.load_state_dict(checkpoint['model_state_dict_net'])
        

Directory /home/dd3/AI_uncert/NEMA-DL/working_dir  already exists!
/home/dd3/AI_uncert/NEMA-DL/working_dir


<All keys matched successfully>

In [23]:

import sirf.STIR as pet
umap = pet.ImageData(NEMApath+'/umap.hv').as_array()
umap_t = torch.from_numpy(umap).repeat(1,1,1,1,1).to(device)
# net_in = make_image_network_compatible(8, umap_t).to(device)
torch.onnx.export(net, umap_t, 'net.onnx', input_names=["umap"], output_names=["recon"])


/home/dd3/devel/github/ARTCertainty/artcertainty/architectures/UNet3D.py:138: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if (shape_it % div!=0 and shape_it!=1 and dim>1):
/home/dd3/devel/github/ARTCertainty/artcertainty/architectures/UNet3D.py:139: TracerWarning: Converting a tensor to a Python integer might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  q = int(shape_it / div)
/home/dd3/devel/github/ARTCertainty/artcertainty/architectures/UNet3D.py:141: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value w

In [ ]:
umap_small = anat.zoom_image(size=(75,80,80))
print(umap_small.shape)
umap_small.write('../umap_smaller.hv')

In [ ]:
# parameters 
hybrid=True # i don't get why kem is not working
deep_hybrid=True
LR = 1e-3
num_subsets=9
num_iter=1
num_deep_iter=200
outer_iter=1
save_every=1
# deep_freeze_at=600
# recon=pet.KOSMAPOSLReconstructor()
sdm=5
ksigma=0.1 #06
k=27
w=5
print(add_sino.max(), sinogram_template.max(), norm_sino.max(),anat.max(), umap.max())
# K=kernelise_image()
# set_KOSMAPOSL(recon, obj_fun, anat, at, hybrid, w,sigma_m,sigma_p, True, sdm )
# uter_iter,em_iter,deep_iter, is2d, w, k, learnin_rate,
#                  ksigma, anat, sinogram_template, umap, norm, 
#                  additive, format, save_every, is_real, psf_fwhm, 
#                  epoch_checkpoint, net_scale, a_seed
algorithm = neuralKEM(outer_iter, 4, 150,
                          0, w, k, LR, ksigma, anat, sinogram_template,
                          umap, norm_sino, add_sino, 'npy', save_every, True,
                          0, 0,0,None)


from artcertainty.kernel.LHK import BuildK
BK=BuildK(algorithm.ksigma,False)
K=BK(algorithm.anatomical_tensor,algorithm.k,algorithm.w)
l = algorithm.run(42)

In [ ]:
del y
del add_sino_tt

# del norm_sino_tt

gc.collect()
torch.cuda.empty_cache()
def get_knn_v(ref, window, k, test=None):
        with torch.no_grad():
            ref=ref.to(ref.device)
            w = window
            # indeces
            num_elements = ref.numel()
            idx=torch.arange((num_elements)).to(ref.device)#.reshape((ref.shape[0],ref.shape[1]),test.shape[2],test.shape[3])
            idx=torch.reshape(idx,(ref.shape))

            # define window and neighbourhood   
            if(ref.shape[1]==1 and ref.dim()==4):
                is3d=False
                query=torch.ones((1,1,w,w)).to(ref.device)
            else:
                is3d=True
                query=torch.ones((1,w,w,w)).to(ref.device)
                ref= ref[0,...]

            # create a big tensor containing value of neighbours for each voxel so each size N becomes w x N
            if (is3d):
                # the following creates a tensor with repeating values for each voxel as if each neighbour was substituting each voxel
                # This is will be usefull to calculate the difference in value between each voxel and its neighbouring voxels
                ext_in = torch.repeat_interleave(ref, repeats=query.shape[1], dim=1).repeat_interleave(repeats=query.shape[2], dim=2).repeat_interleave(repeats=query.shape[3], dim=3)
                unfolded=ref.unfold(3,w,1).unfold(2,w,1).unfold(1,w,1).squeeze(0).squeeze(0).squeeze(0)
            else:
                ext_in = torch.repeat_interleave(ref, repeats=query.shape[2], dim=2).repeat_interleave(repeats=query.shape[3], dim=3)
                unfolded=ref.unfold(3,w,1).unfold(2,w,1).squeeze(0).squeeze(0)
            del ref
            gc.collect()
            torch.cuda.empty_cache()
            # extende unfolded for NN. Something a bit better could be done her but OK for now
            eu=BK.duplicate_last_cols_and_rows(unfolded,w) 

            del unfolded
            gc.collect()
            torch.cuda.empty_cache()

            eu_shape = eu.shape

            if (is3d):
                # NN=eu.view(eu_shape[0],eu_shape[1],1,eu_shape[3],eu_shape[4],eu_shape[2]*eu_shape[5])#.permute(0,1,2,3,5,4)
                NN=eu.permute(0,1,3,4,2, 5)
                NN=NN.permute(0,2,1,3,4, 5)
                NN=NN.reshape(ext_in.shape)
        
            else:
                NN=eu.view(eu_shape[0],1,eu_shape[2]*eu_shape[1],eu_shape[3]).permute(0,1,3,2)
                NN=NN.reshape(ext_in.shape)

            del eu
            gc.collect()
            torch.cuda.empty_cache()

            #same for index
            if (is3d):
                idx = idx[0,...]
                id_nn=idx.unfold(3,w,1).unfold(2,w,1).unfold(1,w,1).squeeze(0).squeeze(0).squeeze(0)
            else:
                id_nn=idx.unfold(3,w,1).unfold(2,w,1).squeeze(0).squeeze(0)

            id_nn=BK.duplicate_last_cols_and_rows(id_nn,w) #extended unfolded for NN

            if (is3d):
                # id_nn=id_nn.view(id_nn.shape[0],id_nn.shape[1],1,id_nn.shape[3],id_nn.shape[4],id_nn.shape[2]*id_nn.shape[5])#.permute(0,1,2,3,5,4)
                id_nn=id_nn.permute(0,1,3,4,2, 5)
                id_nn=id_nn.permute(0,2,1,3,4, 5)
                id_nn=id_nn.reshape(ext_in.shape)   
            else:
                id_nn=id_nn.view(id_nn.shape[0],1,id_nn.shape[2]*id_nn.shape[1],id_nn.shape[3]).permute(0,1,3,2)
                id_nn=id_nn.reshape(ext_in.shape)

            #calculate distance between voxel and it's neighbours
            #reorganise tensor in w*w x N then k x N
            #print(ext_in)    
            if (is3d):
                #  NN= NN.reshape(1,1,num_elements,query.numel())
                NN= NN.unfold(3,w,w).unfold(2,w,w).unfold(1,w,w).reshape(1,1,num_elements,query.numel())
                ext_in=ext_in.unfold(3,w,w).unfold(2,w,w).unfold(1,w,w).reshape(1,1,num_elements,query.numel())
                id_nn=id_nn.unfold(3,w,w).unfold(2,w,w).unfold(1,w,w).reshape(1,1,num_elements,query.numel())
            else:
                NN= NN.unfold(3,w,w).unfold(2,w,w).reshape(1,1,num_elements,query.numel())
                ext_in=ext_in.unfold(3,w,w).unfold(2,w,w).reshape(1,1,num_elements,query.numel())
                id_nn=id_nn.unfold(3,w,w).unfold(2,w,w).reshape(1,1,num_elements,query.numel())

            dist=abs(NN-ext_in)

            del ext_in
            gc.collect()
            torch.cuda.empty_cache()
        
            sorted_dist, indices = torch.sort(dist, dim=-1)

            ID=id_nn.gather(dim=3,index=indices)
            D,IDsr,IDs = sorted_dist[:,:,:, 0:k], ID[:,:,:, 0:k],indices[:,:,:, 0:k]
            #IDt=ID[:,:,,:]
            W=NN.gather(dim=3, index=IDs)
        if test!= None:
            return W, IDsr, id_nn
        else:
            return W, IDsr

W, ID = get_knn_v(algorithm.anatomical_tensor, algorithm.w, algorithm.k, test=None)
    

In [ ]:
is_voxelised=False
import math
def get_K(W,ID,k,ksigma):
    with torch.no_grad():
        
        Kw=torch.zeros(W[0,0,...].shape).to(W.device)
        id=torch.tensor(range(W.shape[2]),  dtype=torch.int64).to(W.device).reshape(1,1,W.shape[2],1)

        id= id.expand(W.shape).reshape(1,1,W.shape[2],W.shape[3],1)
        idx= torch.cat([id, ID.reshape(1,1,W.shape[2],W.shape[3],1)],4)
        sigma=torch.std(W)
        del id
        gc.collect()
        torch.cuda.empty_cache()

        for i in range(W.shape[2]):

            if( sigma== 0):        
                Kw[i,:]=torch.zeros(1,1,1,W.shape[3])
            else:
                if (is_voxelised):
                    norm = W[0,0,i,0]-W[0,0,ID[0,0,i,:],0]
                    Kw[i,:]=(-torch.square((norm/sigma)/(math.sqrt(2)*ksigma)))
                else:
                    norm = torch.nn.functional.pairwise_distance(W[0,0,i,:],W[0,0,ID[0,0,i,:],:],p=2)
                    Kw[i,:]=(-(norm/sigma)/(math.sqrt(2)*ksigma))

            Kw[i,:]=torch.nn.functional.softmax(Kw[i,:], dim = 0)
        # implementation of sparse_coo is apparently very slow in gpu so sending operation on cpu
        idx=idx.reshape(Kw.numel(),2).detach().cpu()
        Kw=Kw.reshape(Kw.numel()).detach().cpu()
        K = torch.sparse_coo_tensor(list(zip(*idx)), Kw,(ID.shape[2],ID.shape[2]))

        return K#.to_sparse()

K = get_K(W,ID,k,algorithm.ksigma)